# NIH ChestX-ray14 + RAG — Kaggle Runner

**Before running, in the right-hand sidebar:**
1. **Add Input** ▸ search `NIH Chest X-rays` ▸ add the dataset by *nih-chest-xrays* (mounts at `/kaggle/input/data`).
2. **Add Input** ▸ Datasets ▸ add your uploaded project (the `nih-chestxray14-rag` repo zipped and uploaded as a Kaggle Dataset).
3. **Accelerator** ▸ **GPU P100** (or T4 x2).
4. **Internet** ▸ **On** (needed for pip + the Gemini call).

Then run top to bottom.

## 1. Confirm GPU + see the mounted data

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!ls /kaggle/input/data | head
# You should see images_001 ... images_012, Data_Entry_2017.csv, train_val_list.txt, test_list.txt

## 2. Copy the repo into the writable working dir
`/kaggle/input` is read-only, so we copy the code to `/kaggle/working` to run + write outputs.
Adjust the source path to match your uploaded dataset's folder name.

In [ ]:
import shutil, glob, os
# Find the repo folder inside your uploaded dataset (contains config.py):
candidates = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/config.py', recursive=True)]
assert candidates, 'Could not find config.py in /kaggle/input — did you add your project dataset?'
src = candidates[0]
print('Found repo at:', src)
shutil.copytree(src, '/kaggle/working/proj', dirs_exist_ok=True)
%cd /kaggle/working/proj
!ls

## 3. Install the extra dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-generativeai mlflow
# torch / torchvision / sklearn / pandas are already on Kaggle.

## 4. Configure
Data path is already `/kaggle/input/data` (the config default). We send outputs to
`/kaggle/working` (persists when you **Save Version**) and set a moderate subsample
that finishes comfortably on a P100. Set the two MAX_* to `None` for the full run.

In [ ]:
import re, pathlib
cfg = pathlib.Path('config.py').read_text()
edits = {
    r'DATA_DIR = .*': 'DATA_DIR = Path("/kaggle/input/data")',
    r'OUTPUT_DIR = .*': 'OUTPUT_DIR = Path("/kaggle/working/outputs")',
    r'BATCH_SIZE = .*': 'BATCH_SIZE = 64',
    r'NUM_WORKERS = .*': 'NUM_WORKERS = 2',
    r'EPOCHS = .*': 'EPOCHS = 8',
    r'MAX_TRAIN_PATIENTS = .*': 'MAX_TRAIN_PATIENTS = 6000   # ~20k images; None = full',
    r'MAX_EVAL_IMAGES = .*': 'MAX_EVAL_IMAGES = 6000        # None = full test set',
}
for pat, rep in edits.items():
    cfg = re.sub(pat, rep, cfg, count=1)
pathlib.Path('config.py').write_text(cfg)
print('config.py updated.')

## 5. Sanity check (no training yet)

In [ ]:
!python run_smoke_test.py

## 6. Train the three models
If VGG-19 OOMs on the P100, drop BATCH_SIZE to 32 in cell 4 and rerun that one.

In [ ]:
!python -m src.train --model resnet18

In [ ]:
!python -m src.train --model vgg19

In [ ]:
!python -m src.train --model customcnn

## 7. Comparison tables + figures

In [ ]:
!python -m src.report_figures
from IPython.display import Image, display
import os
fig = '/kaggle/working/outputs/figures'
for f in ['learning_curves.png', 'auroc_comparison.png']:
    if os.path.exists(f'{fig}/{f}'):
        display(Image(f'{fig}/{f}'))

## 8. Grad-CAM on a sample image

In [ ]:
import glob
sample = sorted(glob.glob('/kaggle/input/data/**/*.png', recursive=True))[0]
!python -m src.gradcam --model resnet18 --image "{sample}" --label Cardiomegaly
from IPython.display import Image, display
import glob as g
cams = g.glob('/kaggle/working/outputs/figures/gradcam_*.png')
if cams: display(Image(cams[0]))

## 9. RAG index + grounded LLM summary (Gemini)

In [ ]:
import os
# Better: store the key in Kaggle 'Add-ons > Secrets' and load it; or paste here.
os.environ['GEMINI_API_KEY'] = 'PASTE_YOUR_GEMINI_KEY'   # aistudio.google.com
!python -m src.rag.build_index
!python -m src.rag.interpret --demo

## Done
Outputs (checkpoints, metrics, figures) are in `/kaggle/working/outputs`.
Click **Save Version** (top-right) to persist them and make them downloadable.
Paste numbers into `report/report_template.md`, then push to GitHub.

**Full-dataset run:** set `MAX_TRAIN_PATIENTS = None`, `MAX_EVAL_IMAGES = None`,
`EPOCHS = 10` in cell 4. Expect a long run — use Save Version (runs up to ~12h)
rather than the interactive session, and the per-model Drive/working checkpoints
let you resume if needed.